# SMT End-to-End Pipeline

This notebook covers:
- Optional web crawling (for assignment evidence)
- Dataset loading (JSON/CSV/TSV/TXT)
- Text cleaning + tokenization
- Statistical Machine Translation training (IBM1-style)
- Saving training progress and checkpoints
- Resume training


In [ ]:
# -*- coding: utf-8 -*-
import os, json, re, random, time, math, csv
from collections import defaultdict, Counter
from typing import List, Tuple, Dict, Set
from datetime import datetime
import time as _time

import pandas as pd

# ---------- Optional installs ----------
try:
    import jieba
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jieba"])
    import jieba

try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

# ==============================
# CONFIG
# ==============================
# Point this to your dataset (CN \t EN per line for .txt)
DATASET_PATH = r"C:\Users\Kevin\Desktop\Artificial Intelligent\Artificial-Intelligent\data\dataset_CN_EN.txt"   # <-- change if needed

# If you use CSV/TSV/JSON with headers, set these columns accordingly.
DATASET_TEXT_COLUMNS = ["chinese", "english"]

RUN_DIR = "./smt_runs/zh_en_pbsmt_s2t_only"

# Data / training caps
MAX_SENTENCES       = 200000
MAX_SENT_LEN        = 40
SEED                = 42

# IBM1
IBM1_ITERS          = 78
IGNORE_STOPWORDS    = False
USE_IDF_WEIGHT      = True
ADD_NULL            = True
DICE_TOPK_PER_TOKEN = 40
DICE_MIN_THRESH     = 0.005

# Phrase extraction
MAX_SRC_PHRASE_LEN  = 3
PHRASE_TOPK_PER_SRC = 20

# LM (Interpolated Kneser–Ney)
LM_ORDER            = 5
DISCOUNT            = 0.75  # fixed discount works ok on small data

# Decoder (with limited reordering)
W_PHRASE            = 1.0
W_LEX               = 0.7
W_LM                = 0.9
W_WORD_PENALTY      = -0.12
MAX_JUMP            = 1
DIST_PENALTY        = -0.25

# Eval
BLEU_SAMPLE_SIZE    = 1000

# Tuning
DO_TUNE             = True
TUNE_TRIALS         = 5
TUNE_DEV_SIZE       = 200
TUNE_FAST_SUBSET    = 80  # per trial for speed

# Checkpoints
RESET_CHECKPOINTS   = True
STATUS_PATH         = os.path.join(RUN_DIR, "status.json")

random.seed(SEED)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(os.path.join(RUN_DIR, "checkpoints"), exist_ok=True)
if RESET_CHECKPOINTS:
    ckdir = os.path.join(RUN_DIR, "checkpoints")
    for fn in os.listdir(ckdir):
        try: os.remove(os.path.join(ckdir, fn))
        except: pass
    for fn in ["status.json", "metrics.csv", "phrase_table.json", "lm_meta.json"]:
        fpath = os.path.join(RUN_DIR, fn)
        if os.path.exists(fpath):
            try: os.remove(fpath)
            except: pass

# ===========
# Overrides (tiny curated dictionary to pin obvious phrases)
# ===========
# Each CN token/phrase (as tuple of jieba tokens) maps to EN token(s)
# These entries are injected both into IBM1 backoff and phrase table with high scores.
OVERRIDES = {
    ("你好",): ("hello",),
    ("世界",): ("world",),
    ("你好", "世界"): ("hello", "world"),
    ("谢谢",): ("thanks",),
}

# ==============================
# Helpers
# ==============================
EN_STOP = set(nltk_stopwords.words("english"))

# seed jieba with common MT-ish words
for w in ["你好","谢谢","世界","中国","我们","学校","喜欢","学习","英文","中文"]:
    jieba.add_word(w)

def tok_zh_words(s: str) -> List[str]:
    return [t for t in jieba.lcut(str(s)) if t.strip()]

def tok_en_words(s: str) -> List[str]:
    toks = re.findall(r"\b\w+\b", str(s).lower())
    return [t for t in toks if (not IGNORE_STOPWORDS or t not in EN_STOP)]

def _smart_txt_split(line: str) -> List[str]:
    for sep in ["\t", " | ", "||", ":::", "|", ",", "  "]:
        if sep in line:
            parts = line.split(sep, 1)
            if len(parts) >= 2:
                return [parts[0], parts[1]]
    parts = line.strip().split(None, 1)
    if len(parts) == 2:
        return parts
    return []

def load_dataset(path: str, text_cols: List[str]) -> Tuple[List[str], List[str]]:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".json", ".jsonl"):
        try:
            df = pd.read_json(path, lines=True)
        except ValueError:
            df = pd.read_json(path)
        if not all(c in df.columns for c in text_cols):
            raise ValueError(f"Columns {text_cols} not found. Available: {list(df.columns)}")
        df = df[text_cols].dropna().head(MAX_SENTENCES).reset_index(drop=True)
        return list(df[text_cols[0]]), list(df[text_cols[1]])
    elif ext in (".csv", ".tsv"):
        sep = "," if ext == ".csv" else "\t"
        df = pd.read_csv(path, sep=sep)
        if not all(c in df.columns for c in text_cols):
            raise ValueError(f"Columns {text_cols} not found. Available: {list(df.columns)}")
        df = df[text_cols].dropna().head(MAX_SENTENCES).reset_index(drop=True)
        return list(df[text_cols[0]]), list(df[text_cols[1]])
    else:
        srcs, tgts = [], []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = _smart_txt_split(line)
                if len(parts) >= 2:
                    srcs.append(parts[0].strip())
                    tgts.append(parts[1].strip())
                if len(srcs) >= MAX_SENTENCES: break
        if not srcs:
            raise ValueError("Could not parse any lines from .txt.")
        return srcs, tgts

def bleu_corpus(refs: List[List[str]], hyps: List[List[str]], max_n=4) -> float:
    def ngrams(seq, n): return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]
    logs = []
    for n in range(1, max_n+1):
        match = total = 0
        for r, h in zip(refs, hyps):
            rc, hc = Counter(ngrams(r, n)), Counter(ngrams(h, n))
            total += sum(hc.values())
            for g, c in hc.items():
                match += min(c, rc.get(g, 0))
        logs.append(float("-inf") if total == 0 or match == 0 else math.log(match/total))
    ref_len = sum(len(r) for r in refs)
    hyp_len = sum(len(h) for h in hyps)
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len/max(hyp_len,1))
    gm = 0.0 if any(x == float("-inf") for x in logs) else math.exp(sum(logs)/len(logs))
    return bp * gm

def append_metrics(row: Dict[str, str]):
    path = os.path.join(RUN_DIR, "metrics.csv")
    write_header = not os.path.exists(path)
    with open(path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=sorted(row.keys()))
        if write_header: w.writeheader()
        w.writerow(row)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _now_iso():
    return datetime.now().isoformat(timespec="seconds")

def update_status(stage: str, it: int, total_iters: int, last_ckpt: str, extra: dict = None):
    try:
        status = {}
        if os.path.exists(STATUS_PATH):
            with open(STATUS_PATH, "r", encoding="utf-8") as f:
                status = json.load(f)
        status[stage] = {
            "current_iter": it,
            "total_iters": total_iters,
            "last_checkpoint": last_ckpt,
            "updated_at": _now_iso()
        }
        if extra:
            status[stage].update(extra)
        with open(STATUS_PATH, "w", encoding="utf-8") as f:
            json.dump(status, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print("[status] WARN:", e)

# ==============================
# Prepare data
# ==============================
src_raw, tgt_raw = load_dataset(DATASET_PATH, DATASET_TEXT_COLUMNS)
print("Loaded", len(src_raw), "pairs")

pairs: List[Tuple[List[str], List[str]]] = []
for z, e in zip(src_raw, tgt_raw):
    sw, tw = tok_zh_words(z), tok_en_words(e)
    if 0 < len(sw) <= MAX_SENT_LEN and 0 < len(tw) <= MAX_SENT_LEN:
        pairs.append((sw, tw))
random.shuffle(pairs)
print("After filters:", len(pairs))

# Build Dice candidate pruning (s->t)
src_df, tgt_df = Counter(), Counter()
pair_df = defaultdict(Counter)
for s_words, t_words in pairs:
    s_set, t_set = set(s_words), set(t_words)
    for s in s_set: src_df[s] += 1
    for t in t_set: tgt_df[t] += 1
    for s in s_set:
        for t in t_set:
            pair_df[s][t] += 1

candidates_s2t: Dict[str, Set[str]] = {}
for s, t_counts in pair_df.items():
    scored = []
    for t, freq in t_counts.items():
        dice = 2.0 * freq / (src_df[s] + tgt_df[t])
        if dice >= DICE_MIN_THRESH:
            scored.append((t, dice))
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates_s2t[s] = set(t for t, _ in scored[:DICE_TOPK_PER_TOKEN])

# Build Dice candidate pruning (t->s)
candidates_t2s: Dict[str, Set[str]] = {}
for s, t_counts in pair_df.items():
    for t, freq in t_counts.items():
        dice = 2.0 * freq / (src_df[s] + tgt_df[t])
        if dice >= DICE_MIN_THRESH:
            candidates_t2s.setdefault(t, set()).add(s)

# IDF for EN (target) and ZH (source)
def build_idf_on(tokens_list: List[List[str]]) -> Dict[str, float]:
    df_c = Counter()
    for toks in tokens_list:
        df_c.update(set(toks))
    N = len(tokens_list)
    idf = defaultdict(lambda: 1.0)
    for w, c in df_c.items():
        idf[w] = math.log(1 + (N / (1 + c))) + 1.0
    return idf

idf_en = build_idf_on([t for _, t in pairs])
idf_zh = build_idf_on([s for s, _ in pairs])

# ==============================
# IBM1 Training (s -> t only)
# ==============================
NULL = "<NULL>"
NULL_PENALTY = 0.5

def train_ibm1_s2t(pairs: List[Tuple[List[str], List[str]]],
                   candidates: Dict[str, Set[str]],
                   iters: int,
                   idf_target: Dict[str, float],
                   add_null=True) -> Dict[str, Dict[str, float]]:
    ckdir = os.path.join(RUN_DIR, "checkpoints")
    os.makedirs(ckdir, exist_ok=True)
    tprobs = defaultdict(lambda: defaultdict(lambda: 1.0))
    start_it = 1
    resumes = sorted([n for n in os.listdir(ckdir) if n.startswith("ibm1_s2t_iter_")])
    if resumes:
        last = resumes[-1]
        tprobs = defaultdict(lambda: defaultdict(float))
        data = load_json(os.path.join(ckdir, last))
        for s, d in data.items():
            for t, p in d.items():
                tprobs[s][t] = float(p)
        start_it = int(re.findall(r"(\d+)", last)[-1]) + 1
        print(f"[IBM1 s2t] Resuming from {last}")
        update_status("ibm1_s2t", start_it-1, iters, last_ckpt=os.path.join(ckdir, last))

    # Seed some probs from OVERRIDES so EM starts sensible on frequent basics
    for f_tuple, e_tuple in OVERRIDES.items():
        if len(f_tuple) == 1 and len(e_tuple) == 1:
            f = f_tuple[0]; e = e_tuple[0]
            tprobs[f][e] = max(tprobs[f][e], 5.0)  # big bump

    for it in range(start_it, iters+1):
        t0 = time.time()
        count = defaultdict(Counter)
        total = defaultdict(float)
        for s_words, t_words in pairs:
            t_set = set(t_words)
            if add_null: t_set = set(t_set) | {NULL}
            for s in s_words:
                cand_t = candidates.get(s, set())
                if add_null: cand_t = set(cand_t) | {NULL}
                t_valid = [t for t in t_set if t in cand_t] or ([NULL] if add_null else [])
                if not t_valid:
                    continue
                weights = {}
                denom = 0.0
                for t in t_valid:
                    w = tprobs[s][t]
                    if USE_IDF_WEIGHT:
                        w *= idf_target[t] if t != NULL else 1.0
                    weights[t] = w
                    denom += w
                if denom == 0.0:
                    eq = 1.0 / len(t_valid)
                    for t in t_valid: weights[t] = eq
                    denom = 1.0
                inv = 1.0 / denom
                for t, w in weights.items():
                    frac = w * inv
                    count[s][t] += frac
                    total[s] += frac

        for s in count:
            s_total = total[s] if total[s] > 0 else 1.0
            for t, c in count[s].items():
                tprobs[s][t] = c / s_total

        # quick intrinsic BLEU proxy (top-1 per src token)
        sample = pairs if len(pairs) <= BLEU_SAMPLE_SIZE else random.sample(pairs, BLEU_SAMPLE_SIZE)
        hyps, refs = [], []
        for s_words, t_words in sample:
            hyp = []
            for s in s_words:
                cands = tprobs.get(s, {})
                if cands:
                    best = max([(tt, p) for tt, p in cands.items() if tt != NULL],
                               key=lambda kv: kv[1], default=(None,0))
                    if best[0]: hyp.append(best[0])
            hyps.append(hyp)
            refs.append(t_words)
        train_bleu = bleu_corpus(refs, hyps)
        append_metrics({"stage": "ibm1_s2t", "iter": it, "bleu": f"{train_bleu:.6f}"})

        ckfile = os.path.join(ckdir, f"ibm1_s2t_iter_{it:03d}.json")
        save_json({s: dict(d) for s, d in tprobs.items()}, ckfile)
        update_status("ibm1_s2t", it, iters, last_ckpt=ckfile, extra={"bleu": round(train_bleu, 6)})
        print(f"[IBM1 s2t] Iter {it}/{iters} | BLEU={train_bleu*100:.2f} | {time.time()-t0:.1f}s")
    return tprobs

tprobs_s2t = train_ibm1_s2t(pairs, candidates_s2t, IBM1_ITERS, idf_en, add_null=ADD_NULL)
save_json({s: dict(d) for s, d in tprobs_s2t.items()}, os.path.join(RUN_DIR, "ibm1_s2t_final.json"))

print("\n=== Training status ===")
print(json.dumps({"ibm1_s2t": {"current_iter": IBM1_ITERS, "total_iters": IBM1_ITERS,
                               "last_checkpoint": os.path.join(RUN_DIR, "checkpoints", f"ibm1_s2t_iter_{IBM1_ITERS:03d}.json"),
                               "bleu":"(see metrics.csv)"}}, ensure_ascii=False, indent=2))

# ==============================
# IBM1 Training (t -> s) for symmetrization
# ==============================
def train_ibm1_t2s(pairs, candidates_t2s, iters, idf_src, add_null=True):
    ckdir = os.path.join(RUN_DIR, "checkpoints")
    os.makedirs(ckdir, exist_ok=True)
    tprobs = defaultdict(lambda: defaultdict(lambda: 1.0))
    start_it = 1
    resumes = sorted([n for n in os.listdir(ckdir) if n.startswith("ibm1_t2s_iter_")])
    if resumes:
        last = resumes[-1]
        tprobs = defaultdict(lambda: defaultdict(float))
        data = load_json(os.path.join(ckdir, last))
        for t, d in data.items():
            for s, p in d.items():
                tprobs[t][s] = float(p)
        start_it = int(re.findall(r"(\d+)", last)[-1]) + 1
        print(f"[IBM1 t2s] Resuming from {last}")
        update_status("ibm1_t2s", start_it-1, iters, last_ckpt=os.path.join(ckdir, last))

    for it in range(start_it, iters+1):
        t0 = time.time()
        count = defaultdict(Counter)
        total = defaultdict(float)
        for s_words, t_words in pairs:
            s_set = set(s_words)
            if add_null: s_set = set(s_set) | {NULL}
            for t in t_words:
                cand_s = candidates_t2s.get(t, set())
                if add_null: cand_s = set(cand_s) | {NULL}
                s_valid = [s for s in s_set if s in cand_s] or ([NULL] if add_null else [])
                if not s_valid:
                    continue
                weights, denom = {}, 0.0
                for s in s_valid:
                    w = tprobs[t][s]
                    if USE_IDF_WEIGHT:
                        w *= idf_src[s] if s != NULL else 1.0
                    weights[s] = w
                    denom += w
                if denom == 0.0:
                    eq = 1.0 / len(s_valid)
                    for s in s_valid: weights[s] = eq
                    denom = 1.0
                inv = 1.0 / denom
                for s, w in weights.items():
                    frac = w * inv
                    count[t][s] += frac
                    total[t] += frac

        for t in count:
            t_total = total[t] if total[t] > 0 else 1.0
            for s, c in count[t].items():
                tprobs[t][s] = c / t_total

        ckfile = os.path.join(ckdir, f"ibm1_t2s_iter_{it:03d}.json")
        save_json({t: dict(d) for t, d in tprobs.items()}, ckfile)
        update_status("ibm1_t2s", it, iters, last_ckpt=ckfile)
        print(f"[IBM1 t2s] Iter {it}/{iters} | {time.time()-t0:.1f}s")
    return tprobs

tprobs_t2s = train_ibm1_t2s(pairs, candidates_t2s, IBM1_ITERS, idf_zh, add_null=ADD_NULL)
save_json({t: dict(d) for t, d in tprobs_t2s.items()}, os.path.join(RUN_DIR, "ibm1_t2s_final.json"))

# ==============================
# Viterbi Alignments both ways
# ==============================
def viterbi_align_one_s2t(s_words, t_words, tprobs):
    aligns = set()
    for i, s in enumerate(s_words):
        best_j, best_p = -1, 0.0
        for j, t in enumerate(t_words):
            p = tprobs.get(s, {}).get(t, 0.0)
            if p > best_p:
                best_p, best_j = p, j
        p_null = tprobs.get(s, {}).get(NULL, 0.0) * NULL_PENALTY
        if p_null >= best_p:
            continue
        if best_j >= 0:
            aligns.add((i, best_j))
    return aligns

def viterbi_align_one_t2s(s_words, t_words, tprobs):
    aligns = set()
    for j, t in enumerate(t_words):
        best_i, best_p = -1, 0.0
        for i, s in enumerate(s_words):
            p = tprobs.get(t, {}).get(s, 0.0)
            if p > best_p:
                best_p, best_i = p, i
        p_null = tprobs.get(t, {}).get(NULL, 0.0) * NULL_PENALTY
        if p_null >= best_p:
            continue
        if best_i >= 0:
            aligns.add((best_i, j))
    return aligns

alignments_s2t = []
alignments_t2s = []
with open(os.path.join(RUN_DIR, "checkpoints", "alignments_t2s.jsonl"), "w", encoding="utf-8") as fout:
    for idx, (s_words, t_words) in enumerate(pairs):
        A_s2t = viterbi_align_one_s2t(s_words, t_words, tprobs_s2t)
        A_t2s = viterbi_align_one_t2s(s_words, t_words, tprobs_t2s)
        alignments_s2t.append(A_s2t)
        alignments_t2s.append(A_t2s)
        fout.write(json.dumps({"idx": idx, "s": s_words, "t": t_words,
                               "a_s2t": sorted(list(A_s2t)),
                               "a_t2s": sorted(list(A_t2s))}, ensure_ascii=False) + "\n")

# ==============================
# grow-diag-final-and symmetrization (per-sentence)
# ==============================
def symmetrize_gdfa(A_s2t: Set[Tuple[int,int]], A_t2s: Set[Tuple[int,int]],
                    I: int, J: int) -> Set[Tuple[int,int]]:
    inter = A_s2t & A_t2s
    union = A_s2t | A_t2s
    res = set(inter)

    def neighbors(i,j):
        for di,dj in [(-1,0),(0,-1),(1,0),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
            ii, jj = i+di, j+dj
            if 0 <= ii < I and 0 <= jj < J:
                yield (ii,jj)

    grow = True
    while grow:
        grow = False
        newly = set()
        for (i,j) in list(res):
            for (ii,jj) in neighbors(i,j):
                if (ii,jj) in union and (ii,jj) not in res:
                    is_aligned_src = ii in {x for x,_ in res}
                    is_aligned_tgt = jj in {y for _,y in res}
                    if not is_aligned_src or not is_aligned_tgt:
                        newly.add((ii,jj))
        if newly:
            res |= newly
            grow = True
    # final
    for (i,j) in (union - res):
        is_aligned_src = i in {x for x,_ in res}
        is_aligned_tgt = j in {y for _,y in res}
        if not is_aligned_src or not is_aligned_tgt:
            res.add((i,j))
    return res

alignments_gdfa = []
ck_align_sym_path = os.path.join(RUN_DIR, "checkpoints", "alignments_sym_gdfa.jsonl")
with open(ck_align_sym_path, "w", encoding="utf-8") as fout:
    for (s_words, t_words), A_s2t, A_t2s in zip(pairs, alignments_s2t, alignments_t2s):
        res = symmetrize_gdfa(A_s2t, A_t2s, len(s_words), len(t_words))
        alignments_gdfa.append(res)
        fout.write(json.dumps({"s": s_words, "t": t_words, "a": sorted(list(res))}, ensure_ascii=False) + "\n")
print("Saved symmetrized alignments:", ck_align_sym_path)

# ==============================
# Phrase Extraction (Koehn-style, using GDFA alignments)
# ==============================
def extract_phrases_for_sentence(s_words, t_words, align_set: Set[Tuple[int,int]], max_src_len=MAX_SRC_PHRASE_LEN):
    phrases = []
    I, J = len(s_words), len(t_words)
    aligned_to_t = defaultdict(set)
    aligned_to_s = defaultdict(set)
    for i,j in align_set:
        aligned_to_s[i].add(j)
        aligned_to_t[j].add(i)

    for i1 in range(I):
        for i2 in range(i1, min(I, i1 + max_src_len)):
            js = [j for i in range(i1, i2+1) for j in aligned_to_s.get(i, [])]
            if not js:
                continue
            j_min, j_max = min(js), max(js)
            # consistency
            out = False
            for j in range(j_min, j_max+1):
                for i in aligned_to_t.get(j, []):
                    if i < i1 or i > i2:
                        out = True; break
                if out: break
            if out: continue

            # expand over unaligned target words
            j1 = j_min
            while j1 >= 0 and (j1 not in aligned_to_t):
                j1 -= 1
            j1 += 1
            j2 = j_max
            while j2 < J and (j2 not in aligned_to_t):
                j2 += 1
            j2 -= 1
            for y1 in range(j1, j_min+1):
                for y2 in range(j_max, j2+1):
                    f = tuple(s_words[i1:i2+1])
                    e = tuple(t_words[y1:y2+1])
                    phrases.append((f, e))
    return phrases

phrase_counts = Counter()
src_phrase_total = Counter()
for (s_words, t_words), align in zip(pairs, alignments_gdfa):
    extracted = extract_phrases_for_sentence(s_words, t_words, align, max_src_len=MAX_SRC_PHRASE_LEN)
    for f, e in extracted:
        phrase_counts[(f, e)] += 1
        src_phrase_total[f] += 1

# Inject OVERRIDES directly as phrases with strong mass
for f_tuple, e_tuple in OVERRIDES.items():
    phrase_counts[(f_tuple, e_tuple)] += 1000
    src_phrase_total[f_tuple] += 1000

# φ(e|f)
phrase_table = defaultdict(lambda: defaultdict(float))
for (f, e), c in phrase_counts.items():
    phrase_table[f][e] = c / max(1, src_phrase_total[f])

# ==============================
# Lexical weights (using IBM1 s->t)
# ==============================
def lexical_weight_e_given_f(e: Tuple[str, ...], f: Tuple[str, ...], tprobs_s2t) -> float:
    prod = 1.0
    for ei in e:
        numer = sum(tprobs_s2t.get(fj, {}).get(ei, 0.0) for fj in f)
        denom = len(f)
        if numer == 0.0:
            numer = tprobs_s2t.get("<NULL>", {}).get(ei, 0.0)
            denom = 1
        prod *= max(numer / max(denom,1), 1e-12)
    return prod

lex_table = defaultdict(lambda: defaultdict(float))
for f, e_dict in phrase_table.items():
    for e in e_dict:
        lex_table[f][e] = lexical_weight_e_given_f(e, f, tprobs_s2t)

# Trim phrase table to top-K per source phrase (by φ * lex)
for f, e_dict in list(phrase_table.items()):
    scored = []
    for e, phi in e_dict.items():
        score = math.log(max(phi, 1e-12)) + 0.5 * math.log(max(lex_table[f][e], 1e-12))
        scored.append((e, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    keep = set([e for e, _ in scored[:PHRASE_TOPK_PER_SRC]])
    phrase_table[f] = {e: phrase_table[f][e] for e in keep}
    lex_table[f]   = {e: lex_table[f][e] for e in keep}

save_json({ "phi": { " ".join(f): { " ".join(e): v for e, v in ed.items() } for f, ed in phrase_table.items() },
            "lex": { " ".join(f): { " ".join(e): v for e, v in ed.items() } for f, ed in lex_table.items() } },
          os.path.join(RUN_DIR, "phrase_table.json"))
print("Saved phrase_table.json")

# Also inject singleton backoff from IBM1 (topk)
BACKOFF_TOPK = 5
for s, d in list(tprobs_s2t.items()):
    if s == NULL: continue
    f = (s,)
    ranked = sorted([(t, p) for t, p in d.items() if t != NULL],
                    key=lambda x: x[1], reverse=True)[:BACKOFF_TOPK]
    if not ranked: continue
    phrase_table.setdefault(f, {})
    lex_table.setdefault(f, {})
    for t, p in ranked:
        e = (t,)
        phi = max(p, 1e-6)
        phrase_table[f][e] = max(phrase_table[f].get(e, 0.0), phi)
        lex_table[f][e] = max(lex_table[f].get(e, 0.0), max(p, 1e-12))

# ==============================
# 5-gram Interpolated Kneser–Ney LM (pure Python)
# ==============================
BOS = "<s>"
EOS = "</s>"

def build_lm_kn(corpus: List[List[str]], order=LM_ORDER):
    counts = [Counter() for _ in range(order)]
    # collect counts
    for toks in corpus:
        seq = [BOS]*(order-1) + toks + [EOS]
        for n in range(1, order+1):
            for i in range(len(seq)-n+1):
                ngram = tuple(seq[i:i+n])
                counts[n-1][ngram] += 1

    # continuation counts for unigrams (number of unique left contexts in bigrams)
    cont_num_unigram = defaultdict(int)
    left_contexts = defaultdict(set)
    for (w1, w2), c in counts[1].items():  # bigrams
        left_contexts[w2].add(w1)
    for w, ctxs in left_contexts.items():
        cont_num_unigram[w] = len(ctxs)

    # denominators for higher orders (sum over followers of hist)
    denom = [defaultdict(int) for _ in range(order)]
    for n in range(2, order+1):
        for ng, c in counts[n-1].items():
            hist = ng[:-1]
            denom[n-1][hist] += c

    def p_continuation_unigram(w):
        # classic KN base
        return cont_num_unigram.get(w, 0) / max(len(counts[1]) or 1, 1)

    # recursive KN with interpolation
    def kn_logprob(nextw, history):
        hist = tuple(history[-(order-1):]) if history else tuple()

        def rec(n, hist, w):
            if n == 1:
                p = p_continuation_unigram(w)
                return math.log(max(p, 1e-12))
            ctable = counts[n-1]
            dtable = denom[n-1]
            ng = hist + (w,)
            c = ctable.get(ng, 0)
            hcount = dtable.get(hist, 0)
            if hcount > 0:
                # distinct followers of hist for alpha mass
                distinct_followers = 0
                for key in ctable:
                    if key[:-1] == hist:
                        distinct_followers += 1
                p_ml = max(c - DISCOUNT, 0.0) / hcount
                alpha = (DISCOUNT * distinct_followers) / hcount
                lower = math.exp(rec(n-1, hist[1:], w))  # backoff on shorter history
                return math.log(max(p_ml + alpha*lower, 1e-12))
            else:
                return rec(n-1, hist[1:], w)

        n_used = min(len(hist)+1, order)
        return rec(n_used, hist[-(n_used-1):], nextw)

    return kn_logprob, {"order": order,
                        "num_unigrams": len(counts[0]),
                        "num_bigrams": len(counts[1]),
                        "num_trigrams": len(counts[2]) if order >=3 else 0}

lm_logprob, lm_meta = build_lm_kn([t for _, t in pairs], order=LM_ORDER)
save_json(lm_meta, os.path.join(RUN_DIR, "lm_meta.json"))
print("Saved LM (Interpolated KN) meta")

# ==============================
# Phrase-based decoder with limited jumps (uses LM history)
# ==============================
src_phrase_index = {}
for f, e_dict in phrase_table.items():
    candidates = []
    for e, phi in e_dict.items():
        lp = math.log(max(phi, 1e-12))
        ll = math.log(max(lex_table[f][e], 1e-12))
        candidates.append((e, lp, ll))
    src_phrase_index[f] = candidates

from functools import lru_cache

def decode_with_jumps(s_words: List[str]) -> List[str]:
    # greedy covering with small jumps + DP over bitmask
    N = len(s_words)
    span_options = defaultdict(list)
    for i in range(N):
        for L in range(1, min(MAX_SRC_PHRASE_LEN, N - i) + 1):
            f = tuple(s_words[i:i+L])
            if f in src_phrase_index:
                span_options[(i, L)] = src_phrase_index[f]

    @lru_cache(maxsize=None)
    def search(mask: int, hist_tuple: Tuple[str, ...]):
        if mask == (1 << N) - 1:
            # EOS
            s_end = lm_logprob(EOS, list(hist_tuple))
            return ([], W_LM * s_end)
        best_hyp, best_score = [], -1e9

        # first uncovered position
        pos = 0
        while pos < N and ((mask >> pos) & 1):
            pos += 1

        advanced = False
        # take a phrase at 'pos'
        for L in range(1, min(MAX_SRC_PHRASE_LEN, N - pos) + 1):
            if any(((mask >> k) & 1) for k in range(pos, pos+L)):
                continue
            if (pos, L) not in span_options:
                continue
            advanced = True
            new_mask = mask | sum(1 << k for k in range(pos, pos+L))
            for e_tokens, lp, ll in span_options[(pos, L)]:
                lm_s = 0.0
                hist = list(hist_tuple)
                for tok in e_tokens:
                    lm_s += lm_logprob(tok, hist)
                    hist = (hist + [tok])[-(LM_ORDER-1):]
                sub_hyp, sub_score = search(new_mask, tuple(hist))
                score = sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s + W_WORD_PENALTY*len(e_tokens)
                if score > best_score:
                    best_score = score
                    best_hyp = list(e_tokens) + sub_hyp

        # allow small jumps for reordering
        for jump in range(1, MAX_JUMP + 1):
            jpos = pos + jump
            if jpos >= N: break
            if (mask >> jpos) & 1:
                continue
            for L in range(1, min(MAX_SRC_PHRASE_LEN, N - jpos) + 1):
                if any(((mask >> k) & 1) for k in range(jpos, jpos+L)):
                    continue
                if (jpos, L) not in span_options:
                    continue
                advanced = True
                new_mask = mask | sum(1 << k for k in range(jpos, jpos+L))
                for e_tokens, lp, ll in span_options[(jpos, L)]:
                    lm_s = 0.0
                    hist = list(hist_tuple)
                    for tok in e_tokens:
                        lm_s += lm_logprob(tok, hist)
                        hist = (hist + [tok])[-(LM_ORDER-1):]
                    sub_hyp, sub_score = search(new_mask, tuple(hist))
                    score = (sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s +
                             W_WORD_PENALTY*len(e_tokens) + DIST_PENALTY*jump)
                    if score > best_score:
                        best_score = score
                        best_hyp = list(e_tokens) + sub_hyp

        # backoff: word-by-word using IBM1 best
        if not advanced and pos < N:
            s = s_words[pos]
            cands = tprobs_s2t.get(s, {})
            best = None
            for t, p in sorted(cands.items(), key=lambda kv: kv[1], reverse=True):
                if t != NULL:
                    best = (t, p); break
            if best:
                tok = best[0]
                lm_s = lm_logprob(tok, list(hist_tuple))
                new_hist = (list(hist_tuple) + [tok])[-(LM_ORDER-1):]
                sub_hyp, sub_score = search(mask | (1 << pos), tuple(new_hist))
                lp = math.log(max(best[1], 1e-12))
                ll = lp
                score = sub_score + W_PHRASE*lp + W_LEX*ll + W_LM*lm_s + W_WORD_PENALTY
                if score > best_score:
                    best_score = score
                    best_hyp = [tok] + sub_hyp

        return (best_hyp, best_score)

    start_hist = tuple([BOS]*(LM_ORDER-1))
    hyp, _ = search(0, start_hist)
    return hyp

# ==============================
# Simple weight tuning via random search (dev BLEU)
# ==============================
# ==============================
# Simple weight tuning via random search (dev BLEU) - VERBOSE
# ==============================
if DO_TUNE:
    DEV_SIZE = min(TUNE_DEV_SIZE, len(pairs)//10 or 1)
    dev = pairs[:DEV_SIZE] if DEV_SIZE > 0 else pairs[:min(500, len(pairs))]
    print(f"[TUNE] dev={len(dev)} | trials={TUNE_TRIALS}", flush=True)

    def decode_bleu_for_weights(Wp, Wl, Wlm, Wwp, Wdist, sample_pairs):
        global W_PHRASE, W_LEX, W_LM, W_WORD_PENALTY, DIST_PENALTY
        W_PHRASE, W_LEX, W_LM, W_WORD_PENALTY, DIST_PENALTY = Wp, Wl, Wlm, Wwp, Wdist
        hyps, refs = [], []
        for s_words, t_words in sample_pairs:
            hyps.append(decode_with_jumps(s_words))
            refs.append(t_words)
        return bleu_corpus(refs, hyps)

    import time as _time
    progress_path = os.path.join(RUN_DIR, "tune_progress.jsonl")
    stop_file = os.path.join(RUN_DIR, "STOP_TUNING")
    TUNE_PATIENCE = 5  # early-stop if no improvement for N trials

    best = None
    stall = 0
    dev_fast = dev[:min(TUNE_FAST_SUBSET, len(dev))] if len(dev) else []
    for i in range(TUNE_TRIALS):
        t0 = _time.perf_counter()
        print(f"[TUNE] trial {i+1}/{TUNE_TRIALS} ...", flush=True)
        # allow external stop
        if os.path.exists(stop_file):
            print("[TUNE] STOP_TUNING detected. Exiting tuning early.", flush=True)
            break

        # sample weights
        Wp  = random.uniform(0.6, 1.6)
        Wl  = random.uniform(0.3, 1.2)
        Wlm = random.uniform(0.8, 1.8)
        Wwp = random.uniform(-0.4, 0.0)
        Wdt = random.uniform(-0.6, -0.05)

        size = len(dev_fast) if dev_fast else len(dev)
        b = decode_bleu_for_weights(Wp, Wl, Wlm, Wwp, Wdt, dev_fast or dev)
        dt = _time.perf_counter() - t0
        print(f"[TUNE] trial {i+1} done  BLEU={b*100:.2f}  (dev={len(dev_fast) or len(dev)} sents)  ({dt:.1f}s)", flush=True)
        dur = _time.perf_counter() - t0

        # log heartbeat line you can tail while it runs
        hb = {"trial": i+1, "trials": TUNE_TRIALS, "dev_size": size,
              "bleu": round(b, 6), "seconds": round(dur, 2)}
        with open(progress_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(hb, ensure_ascii=False) + "\n")

        improved = (best is None) or (b > best[0])
        tag = "NEW BEST" if improved else f"no improve ({stall+1}/{TUNE_PATIENCE})"
        print(f"[TUNE] {i+1}/{TUNE_TRIALS}  BLEU={b*100:.2f}  on {size} sents  ({dur:.1f}s)  {tag}", flush=True)

        if improved:
            best = (b, (Wp, Wl, Wlm, Wwp, Wdt))
            stall = 0
        else:
            stall += 1
            if stall >= TUNE_PATIENCE:
                print("[TUNE] Early stopping (patience reached).", flush=True)
                break

    # lock in best weights
    if best:
        W_PHRASE, W_LEX, W_LM, W_WORD_PENALTY, DIST_PENALTY = best[1]
        append_metrics({"stage": "tuning_random", "iter": 0,
                        "W_PHRASE": f"{W_PHRASE:.3f}", "W_LEX": f"{W_LEX:.3f}",
                        "W_LM": f"{W_LM:.3f}", "W_WORD_PENALTY": f"{W_WORD_PENALTY:.3f}",
                        "DIST_PENALTY": f"{DIST_PENALTY:.3f}", "dev_bleu": f"{best[0]:.6f}"})
        print("[TUNE] locked weights:", best[1], flush=True)


# ==============================
# Evaluation (BLEU on a sample)
# ==============================
sample = pairs if len(pairs) <= BLEU_SAMPLE_SIZE else random.sample(pairs, BLEU_SAMPLE_SIZE)
hyps, refs = [], []
t0 = time.time()
for s_words, t_words in sample:
    hyp = decode_with_jumps(s_words)
    hyps.append(hyp)
    refs.append(t_words)
bleu = bleu_corpus(refs, hyps)
append_metrics({"stage": "pbsmt_decode_jump_sym", "iter": 0, "bleu": f"{bleu:.6f}"})
print(f"[PBSMT+jump sym] BLEU={bleu*100:.2f} on {len(sample)} sents | {time.time()-t0:.1f}s")

# ==============================
# Demos
# ==============================
def topk_word_translations(src_tokens: List[str], tprobs: Dict[str, Dict[str, float]], k=5):
    print("\n=== Top-k word translation points (t(e|f)) ===")
    for s in src_tokens:
        cands = tprobs.get(s, {})
        ranked = sorted([(t, p) for t, p in cands.items() if t != NULL], key=lambda x: x[1], reverse=True)[:k]
        print(f"{s:>10} -> ", ", ".join([f'{t}:{p:.3f}' for t,p in ranked]) or "(none)")

def translate_and_explain(zh_sent: str):
    s_words = tok_zh_words(zh_sent)
    en = decode_with_jumps(s_words)
    print("\nZH:", zh_sent)
    print("EN:", " ".join(en))
    topk_word_translations(s_words, tprobs_s2t, k=5)

for demo in ["你好世界", "谢谢", "今天天气很好", "我们去学校", "我喜欢学习英文", "中国文化很有趣"]:
    translate_and_explain(demo)

print("\nArtifacts saved in:", RUN_DIR)
print(" - IBM1 checkpoints: RUN_DIR/checkpoints/ibm1_s2t_iter_XXX.json & ibm1_t2s_iter_XXX.json")
print(" - Alignments (s2t/t2s): RUN_DIR/checkpoints/alignments_t2s.jsonl")
print(" - Alignments (sym GDFA): RUN_DIR/checkpoints/alignments_sym_gdfa.jsonl")
print(" - Phrase table: RUN_DIR/phrase_table.json")
print(" - LM meta (KN 5-gram): RUN_DIR/lm_meta.json")
print(" - Metrics CSV: RUN_DIR/metrics.csv")
print(" - Status file: RUN_DIR/status.json")


Loaded 20289 pairs
After filters: 20289
[IBM1 s2t] Iter 1/78 | BLEU=4.47 | 1.1s
[IBM1 s2t] Iter 2/78 | BLEU=4.93 | 1.1s
[IBM1 s2t] Iter 3/78 | BLEU=4.77 | 1.2s
[IBM1 s2t] Iter 4/78 | BLEU=4.67 | 1.1s
[IBM1 s2t] Iter 5/78 | BLEU=4.86 | 1.1s
[IBM1 s2t] Iter 6/78 | BLEU=4.45 | 1.1s
[IBM1 s2t] Iter 7/78 | BLEU=5.20 | 1.1s
[IBM1 s2t] Iter 8/78 | BLEU=4.12 | 1.1s
[IBM1 s2t] Iter 9/78 | BLEU=4.54 | 1.3s
[IBM1 s2t] Iter 10/78 | BLEU=5.45 | 1.1s
[IBM1 s2t] Iter 11/78 | BLEU=4.68 | 1.1s
[IBM1 s2t] Iter 12/78 | BLEU=3.55 | 1.1s
[IBM1 s2t] Iter 13/78 | BLEU=4.50 | 1.1s
[IBM1 s2t] Iter 14/78 | BLEU=4.85 | 1.1s
[IBM1 s2t] Iter 15/78 | BLEU=4.77 | 1.1s
[IBM1 s2t] Iter 16/78 | BLEU=4.80 | 1.3s
[IBM1 s2t] Iter 17/78 | BLEU=4.47 | 1.1s
[IBM1 s2t] Iter 18/78 | BLEU=4.73 | 1.1s
[IBM1 s2t] Iter 19/78 | BLEU=4.65 | 1.1s
[IBM1 s2t] Iter 20/78 | BLEU=4.64 | 1.1s
[IBM1 s2t] Iter 21/78 | BLEU=5.19 | 1.1s
[IBM1 s2t] Iter 22/78 | BLEU=4.02 | 1.1s
[IBM1 s2t] Iter 23/78 | BLEU=4.90 | 1.3s
[IBM1 s2t] Iter 24/78 | BL

Traceback (most recent call last):
  File "C:\Users\Kevin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Kevin\AppData\Local\Temp\ipykernel_21248\3020431847.py", line 874, in <module>
    b = decode_bleu_for_weights(Wp, Wl, Wlm, Wwp, Wdt, dev_fast or dev)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Kevin\AppData\Local\Temp\ipykernel_21248\3020431847.py", line 846, in decode_bleu_for_weights
    hyps.append(decode_with_jumps(s_words))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Kevin\AppData\Local\Temp\ipykernel_21248\3020431847.py", line 827, in decode_with_jumps
    hyp, _ = search(0, start_hist)
             ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Kevin\AppData\Local\Temp\ipykernel_21248\3020431847.py", line 772, in search

In [12]:
# ==============================
# Interactive Chinese → English prompt (and a simple function)
# ==============================

def zh2en(zh_text: str) -> str:
    """Translate a Chinese sentence to English using the trained PBSMT decoder."""
    s_words = tok_zh_words(zh_text)
    en = decode_with_jumps(s_words)
    return " ".join(en)

def translate_prompt(show_points: bool = True, k: int = 5):
    """
    Start a small REPL to translate Chinese to English.
    Commands:
      /q            quit
      /k N          set top-k for word translation points (default 5)
      /nop          hide word translation points
      /p            show word translation points
    """
    print("Chinese → English translator. Type Chinese and press Enter.")
    print("Commands: /q (quit), /k N (set top-k), /nop (hide points), /p (show points)\n")
    while True:
        try:
            zh = input("ZH> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye.")
            break

        if not zh:
            continue
        if zh in ("/q", "/quit", "/exit"):
            print("Bye.")
            break
        if zh.startswith("/k"):
            parts = zh.split()
            if len(parts) == 2 and parts[1].isdigit():
                k = max(1, int(parts[1]))
                print(f"Top-k now = {k}")
            else:
                print("Usage: /k N   (example: /k 10)")
            continue
        if zh == "/nop":
            show_points = False
            print("Will hide word translation points.")
            continue
        if zh == "/p":
            show_points = True
            print("Will show word translation points.")
            continue

        # translate
        s_words = tok_zh_words(zh)
        en_words = decode_with_jumps(s_words)
        print("EN>", " ".join(en_words))

        # optional per-word top-k translation points
        if show_points:
            print("\n=== Top-k word translation points (t(e|f)) ===")
            for s in s_words:
                cands = tprobs_s2t.get(s, {})
                ranked = sorted(
                    [(t, p) for t, p in cands.items() if t != "<NULL>"],
                    key=lambda x: x[1],
                    reverse=True
                )[:k]
                if ranked:
                    print(f"{s:>10} -> " + ", ".join([f"{t}:{p:.3f}" for t, p in ranked]))
                else:
                    print(f"{s:>10} -> (none)")
            print()

# --- quick examples ---
#print(zh2en("我们去学校"))
#print(zh2en("今天天气很好"))

# --- start interactive prompt ---
translate_prompt()


Chinese → English translator. Type Chinese and press Enter.
Commands: /q (quit), /k N (set top-k), /nop (hide points), /p (show points)

Bye.


In [2]:
jieba.del_word("今天天气")